# 팀 전체 OOF 종합 비교 (우리 파이프라인 + 김영혜 v5 파이프라인)

**사전 준비**: 다음 9개 파일이 모두 `blend_cache/` 폴더에 있어야 합니다.

우리 쪽 (main.py/실험 노트북들에서 생성):
- `oof_lgbm.npy`, `oof_catboost.npy`, `oof_tt.npy`, `oof_y.npy`

김영혜 v5 쪽 (train_v5.py에서 생성, `attempt file/팀원파일/김영혜/`에서 복사해오기):
- `v5_lgb_oof.npy`, `v5_xgb_oof.npy`, `v5_cat_oof.npy`, `v5_ensemble_oof.npy`, `v5_y.npy`

numpy/sklearn만 쓰는 가벼운 노트북이라 라이브러리 충돌이나 속도 문제는 없습니다.

## 1. 데이터 로드 및 정합성 확인

In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score

CACHE_DIR = "blend_cache"
V5_DIR = "팀원파일/김영혜"

# 우리 쪽
oof_lgbm = np.load(f"{CACHE_DIR}/oof_lgbm.npy")
oof_catboost = np.load(f"{CACHE_DIR}/oof_catboost.npy")
oof_tt = np.load(f"{CACHE_DIR}/oof_tt.npy")
y_ours = np.load(f"{CACHE_DIR}/oof_y.npy")

# 김영혜 v5 쪽
v5_lgb = np.load(f"{V5_DIR}/v5_lgb_oof.npy")
v5_xgb = np.load(f"{V5_DIR}/v5_xgb_oof.npy")
v5_cat = np.load(f"{V5_DIR}/v5_cat_oof.npy")
v5_ens = np.load(f"{V5_DIR}/v5_ensemble_oof.npy")
y_v5 = np.load(f"{V5_DIR}/v5_y.npy")

print(f"우리 쪽 행 수: {len(y_ours)}, v5 쪽 행 수: {len(y_v5)}")

# 가장 중요한 정합성 체크: 두 y가 완전히 같은 행 순서/같은 값인지
if len(y_ours) != len(y_v5):
    print("경고: 행 수가 다릅니다! 두 파이프라인이 다른 데이터를 쓴 것 같습니다.")
else:
    n_diff = np.sum(y_ours.astype(int) != y_v5.astype(int))
    if n_diff == 0:
        print("정합성 확인 완료: 두 y 배열이 완전히 일치합니다. 직접 비교/블렌딩 가능합니다.")
    else:
        print(f"경고: y값이 다른 행이 {n_diff}건 있습니다. 행 순서가 다를 수 있으니 블렌딩 결과를 신중하게 해석하세요.")

y = y_ours

우리 쪽 행 수: 256351, v5 쪽 행 수: 256351
정합성 확인 완료: 두 y 배열이 완전히 일치합니다. 직접 비교/블렌딩 가능합니다.


## 2. 전체 OOF 점수 한눈에 보기

In [3]:
models = {
    "우리: LightGBM (튜닝)": oof_lgbm,
    "우리: CatBoost (재튜닝)": oof_catboost,
    "우리: TabTransformer": oof_tt,
    "v5: LightGBM": v5_lgb,
    "v5: XGBoost": v5_xgb,
    "v5: CatBoost": v5_cat,
    "v5: 앙상블(0.4/0.3/0.3)": v5_ens,
}

scores = {}
for name, pred in models.items():
    score = roc_auc_score(y, pred)
    scores[name] = score
    print(f"{name:30s} AUC: {score:.5f}")

best_single_name = max(scores, key=scores.get)
print(f"\n현재 최고 단독: {best_single_name} ({scores[best_single_name]:.5f})")

우리: LightGBM (튜닝)              AUC: 0.73925
우리: CatBoost (재튜닝)             AUC: 0.73971
우리: TabTransformer             AUC: 0.73523
v5: LightGBM                   AUC: 0.74000
v5: XGBoost                    AUC: 0.74016
v5: CatBoost                   AUC: 0.74007
v5: 앙상블(0.4/0.3/0.3)           AUC: 0.74043

현재 최고 단독: v5: 앙상블(0.4/0.3/0.3) (0.74043)


## 3. 상관계수 매트릭스 (어느 쌍이 진짜 다양성이 있는지)

In [4]:
import itertools

names = list(models.keys())
print(f"{'':30s}", end="")
for n in names:
    print(f"{n[:12]:>14s}", end="")
print()

for n1 in names:
    print(f"{n1:30s}", end="")
    for n2 in names:
        corr = np.corrcoef(models[n1], models[n2])[0, 1]
        print(f"{corr:14.4f}", end="")
    print()

print("\n참고: 1.0에 가까울수록 같은 정보, 낮을수록(0.95 이하) 진짜 다양성이 있어 블렌딩 여지가 있습니다.")

                                우리: LightGBM  우리: CatBoost  우리: TabTrans  v5: LightGBM   v5: XGBoost  v5: CatBoost  v5: 앙상블(0.4/
우리: LightGBM (튜닝)                     1.0000        0.9912        0.9686        0.9892        0.9900        0.9725        0.9867
우리: CatBoost (재튜닝)                    0.9912        1.0000        0.9722        0.9873        0.9890        0.9728        0.9858
우리: TabTransformer                    0.9686        0.9722        1.0000        0.9710        0.9719        0.9616        0.9710
v5: LightGBM                          0.9892        0.9873        0.9710        1.0000        0.9978        0.9870        0.9980
v5: XGBoost                           0.9900        0.9890        0.9719        0.9978        1.0000        0.9877        0.9980
v5: CatBoost                          0.9725        0.9728        0.9616        0.9870        0.9877        1.0000        0.9948
v5: 앙상블(0.4/0.3/0.3)                  0.9867        0.9858        0.9710        0.9980        0.9

## 4. 가장 유망한 쌍들 블렌딩 그리드서치

In [5]:
def grid_search_blend(name_a, pred_a, name_b, pred_b):
    score_a = roc_auc_score(y, pred_a)
    score_b = roc_auc_score(y, pred_b)
    corr = np.corrcoef(pred_a, pred_b)[0, 1]
    baseline = max(score_a, score_b)

    best_w, best_score = (1.0 if score_a >= score_b else 0.0), baseline
    for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        blend = w * pred_a + (1 - w) * pred_b
        blend_score = roc_auc_score(y, blend)
        if blend_score > best_score:
            best_w, best_score = w, blend_score

    improvement = best_score - baseline
    print(f"[{name_a}] vs [{name_b}]")
    print(f"  단독: {score_a:.5f} / {score_b:.5f}  |  상관계수: {corr:.4f}")
    print(f"  최적 블렌딩: {name_a} {best_w:.1f} + {name_b} {1-best_w:.1f} = {best_score:.5f}  (개선: {improvement:+.5f})")
    print()
    return best_w, best_score, improvement


# 우리 최고 모델(LightGBM) vs v5의 각 모델/앙상블
pairs_to_test = [
    ("우리: LightGBM (튜닝)", "v5: 앙상블(0.4/0.3/0.3)"),
    ("우리: LightGBM (튜닝)", "v5: XGBoost"),
    ("우리: CatBoost (재튜닝)", "v5: 앙상블(0.4/0.3/0.3)"),
    ("우리: LightGBM (튜닝)", "v5: LightGBM"),
]

results = []
for name_a, name_b in pairs_to_test:
    w, score, imp = grid_search_blend(name_a, models[name_a], name_b, models[name_b])
    results.append((name_a, name_b, w, score, imp))

best_pair = max(results, key=lambda x: x[4])
print("=" * 60)
print(f"가장 좋은 조합: {best_pair[0]} + {best_pair[1]}")
print(f"가중치: {best_pair[2]:.1f} / {1-best_pair[2]:.1f}  |  점수: {best_pair[3]:.5f}  |  개선: {best_pair[4]:+.5f}")
if best_pair[4] > 0.001:
    print("=> 노이즈를 넘는 의미 있는 개선입니다! 제출 후보로 검토해보세요.")
else:
    print("=> 여전히 노이즈 수준입니다. 두 팀의 파이프라인도 충분히 다양하지 않은 것 같습니다.")

[우리: LightGBM (튜닝)] vs [v5: 앙상블(0.4/0.3/0.3)]
  단독: 0.73925 / 0.74043  |  상관계수: 0.9867
  최적 블렌딩: 우리: LightGBM (튜닝) 0.2 + v5: 앙상블(0.4/0.3/0.3) 0.8 = 0.74050  (개선: +0.00007)

[우리: LightGBM (튜닝)] vs [v5: XGBoost]
  단독: 0.73925 / 0.74016  |  상관계수: 0.9900
  최적 블렌딩: 우리: LightGBM (튜닝) 0.3 + v5: XGBoost 0.7 = 0.74032  (개선: +0.00016)

[우리: CatBoost (재튜닝)] vs [v5: 앙상블(0.4/0.3/0.3)]
  단독: 0.73971 / 0.74043  |  상관계수: 0.9858
  최적 블렌딩: 우리: CatBoost (재튜닝) 0.3 + v5: 앙상블(0.4/0.3/0.3) 0.7 = 0.74055  (개선: +0.00012)

[우리: LightGBM (튜닝)] vs [v5: LightGBM]
  단독: 0.73925 / 0.74000  |  상관계수: 0.9892
  최적 블렌딩: 우리: LightGBM (튜닝) 0.4 + v5: LightGBM 0.6 = 0.74025  (개선: +0.00025)

가장 좋은 조합: 우리: LightGBM (튜닝) + v5: LightGBM
가중치: 0.4 / 0.6  |  점수: 0.74025  |  개선: +0.00025
=> 여전히 노이즈 수준입니다. 두 팀의 파이프라인도 충분히 다양하지 않은 것 같습니다.
